## Lecture guidée du notebook

Cette version annotée a un seul objectif: rendre le notebook compréhensible même si on découvre le projet.

Comment lire cette version:
- chaque cellule importante est précédée d'une explication de son rôle ;
- dans les cellules de code, presque chaque ligne utile est précédée d'un commentaire ;
- les commentaires expliquent non seulement *ce que fait la ligne*, mais surtout *pourquoi elle existe* dans la logique du solveur.

Idée générale du projet:
- on modélise le pilotage d'une centrale sur 168 heures ;
- on maximise le profit via programmation dynamique ;
- on enrichit progressivement l'état pour traiter les extensions ;
- on compare ensuite les scénarios et on exporte les résultats.


# Projet 3 - Dispatch d'une centrale

Notebook autonome pour le problème de base et les extensions.

Contenu :
- chargement robuste des données locales du repo ;
- validation sur le cas 24 heures ;
- résolution du problème de base sur 168 heures ;
- tableaux des décisions optimales et fonctions de valeur ;
- planning on/off et synthèse des résultats ;
- extensions 1, 2 et 3 ;
- export automatique vers `dispatch_results.xlsx`.


Cette introduction pose le cadre global: un seul notebook, mais plusieurs niveaux d'analyse allant du benchmark au cas stochastique.


## Cellule 1 — Initialisation de l'environnement

Cette cellule prépare tout le notebook.

Ce qu'elle fait:
- importe les bibliothèques utiles ;
- définit les chemins vers les fichiers Excel ;
- vérifie que les fichiers requis existent ;
- configure l'affichage Pandas ;
- pose quelques constantes globales (jours, valeur `NEG_INF`).

Pourquoi c'est important:
- toute la suite du notebook dépend de ces chemins et de ces constantes ;
- si un fichier manque, on veut échouer immédiatement plutôt qu'avoir des erreurs floues plus loin.


In [1]:
# Import de `Path` pour manipuler proprement les chemins de fichiers locaux.
from pathlib import Path

# Import de NumPy, utilisé pour les tableaux numériques et les calculs rapides.
import numpy as np
# Import de Pandas, utilisé pour lire les fichiers Excel et construire les tableaux de sortie.
import pandas as pd
# Import d'outils d'affichage pour présenter joliment les résultats dans le notebook.
from IPython.display import Markdown, display

# On fixe le dossier courant comme racine de travail du notebook.
ROOT = Path.cwd()
# Chemin vers le fichier Excel contenant les données hebdomadaires principales.
DATA_XLS = ROOT / "Project3data.xls"
# Chemin vers le fichier de validation 24h fourni avec le projet.
VALIDATION_XLSM = ROOT / "Project3code_Student.xlsm"
# Chemin du fichier Excel qui recevra les résultats exportés à la fin.
OUTPUT_XLSX = ROOT / "dispatch_results.xlsx"

# On échoue immédiatement si le fichier de données principal n'est pas présent.
assert DATA_XLS.exists(), f"Missing file: {DATA_XLS}"
# On vérifie aussi que le fichier de validation existe avant d'aller plus loin.
assert VALIDATION_XLSM.exists(), f"Missing file: {VALIDATION_XLSM}"

# On élargit l'affichage Pandas pour éviter de tronquer les grands tableaux.
pd.set_option("display.max_columns", 200)
# On augmente la largeur d'affichage pour rendre les sorties plus lisibles.
pd.set_option("display.width", 200)
# Liste des jours utilisée dans les tableaux hebdomadaires de restitution.
DAYS = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]
# Grande pénalité négative utilisée pour représenter un état ou une transition impossible.
NEG_INF = -1e15

# Affiche le dossier de travail pour confirmer le contexte d'exécution.
print(f"Working directory: {ROOT}")
# Affiche le nom du fichier de données utilisé.
print(f"Data file: {DATA_XLS.name}")
# Affiche le nom du fichier de validation utilisé.
print(f"Validation file: {VALIDATION_XLSM.name}")


Working directory: c:\Users\3733KN\Documents\Perso\realoption\realoption-p3
Data file: Project3data.xls
Validation file: Project3code_Student.xlsm


## Cellule 2 — Fonctions utilitaires et solveurs

C'est le coeur technique du notebook.

On y trouve:
- le chargement des données ;
- le chargement du cas de validation 24h ;
- le solveur du problème de base ;
- les solveurs des extensions ;
- les fonctions de reconstruction de planning ;
- les helpers de reporting.

À retenir:
- les solveurs suivent tous une logique de Bellman backward ;
- chaque extension ajoute une dimension d'état, pas un nouveau paradigme de calcul.


## Fonctions utilitaires et solveurs DP

Les fonctions ci-dessous reprennent la logique correcte du solveur, avec des chemins locaux et des sorties structurées pour le notebook.


Lis cette partie comme une boîte à outils: chaque fonction isole une étape bien précise de la chaîne de calcul.


In [2]:
# ---------------------------------------------------------------------------
# Fonction 1 — Chargement des données hebdomadaires et stochastiques
# ---------------------------------------------------------------------------
def load_data(filepath=DATA_XLS):
    # On lit la feuille Excel `WeeklySchedule` telle quelle, sans supposer de noms de colonnes.
    # Le notebook travaille ensuite par positions de cellules (`iloc`), car le fichier source
    # est structuré comme une grille projet plutôt qu'une table proprement labellisée.
    df = pd.read_excel(filepath, sheet_name="WeeklySchedule", header=None)

    # On préalloue trois vecteurs de taille 168 = 7 jours * 24 heures.
    # `margins[t]` contiendra la marge unitaire de l'heure t.
    margins = np.zeros(168)
    # `demands[t]` contiendra la demande à servir à l'heure t.
    demands = np.zeros(168)
    # `startups[t]` contiendra le coût de démarrage si la centrale s'allume à l'heure t.
    startups = np.zeros(168)

    # `day_idx` parcourt les 7 colonnes correspondant aux jours de la semaine.
    for day_idx in range(7):
        # `hour_idx` parcourt les 24 heures de la journée courante.
        for hour_idx in range(24):
            # Dans le fichier Excel, les données commencent à la ligne 2 (index Python 2),
            # donc on décale l'heure de +2 pour retrouver la bonne ligne.
            row = hour_idx + 2
            # On transforme le couple (jour, heure) en un indice linéaire `t` entre 0 et 167.
            # C'est ce format aplati qui sera utilisé par tous les solveurs dynamiques.
            t = day_idx * 24 + hour_idx
            # La marge du jour `day_idx` est stockée dans les colonnes 1..7.
            margins[t] = float(df.iloc[row, 1 + day_idx])
            # La demande du jour `day_idx` est stockée plus loin, dans les colonnes 10..16.
            demands[t] = float(df.iloc[row, 10 + day_idx])
            # Les coûts de démarrage du jour `day_idx` sont encore plus loin, colonnes 19..25.
            startups[t] = float(df.iloc[row, 19 + day_idx])

    # On lit ensuite la feuille séparée dédiée à la demande stochastique du dimanche.
    df_stoch = pd.read_excel(filepath, sheet_name="StochasticDemand", header=None)

    # Cette liste contiendra 24 tuples, un par heure du dimanche.
    stoch_demand = []

    # On parcourt les 24 heures du dimanche stochastique.
    for hour_idx in range(24):
        # Même convention de ligne: les valeurs commencent à la ligne Excel 3,
        # donc index Python 2.
        row = hour_idx + 2
        # Premier niveau de demande possible à cette heure.
        D1 = float(df_stoch.iloc[row, 1])
        # Probabilité associée au premier niveau de demande.
        p1 = float(df_stoch.iloc[row, 2])
        # Second niveau de demande possible à cette heure.
        D2 = float(df_stoch.iloc[row, 3])
        # Probabilité associée au second niveau de demande.
        p2 = float(df_stoch.iloc[row, 4])
        # On stocke explicitement les 4 informations pour l'heure courante.
        stoch_demand.append((D1, p1, D2, p2))

    # La fonction renvoie toutes les données nécessaires aux solveurs:
    # les 3 séries hebdomadaires déterministes et le dimanche stochastique.
    return margins, demands, startups, stoch_demand


# ---------------------------------------------------------------------------
# Fonction 2 — Chargement du benchmark 24h servant de validation
# ---------------------------------------------------------------------------
def load_24h_validation(filepath=VALIDATION_XLSM):
    # On lit la feuille `24hours` du fichier de référence fourni par le projet.
    df = pd.read_excel(filepath, sheet_name="24hours", header=None)

    # `margins_24` recevra les 24 marges horaires du cas test.
    margins_24 = np.zeros(24)
    # Dans le benchmark, la demande est constante à 100 pour les 24 heures.
    demands_24 = np.full(24, 100.0)
    # Dans le benchmark, le coût de démarrage est constant à 400.
    startups_24 = np.full(24, 400.0)

    # Les vecteurs de référence ont une taille 25 et non 24,
    # car ils incluent aussi la ligne terminale du Bellman.
    ref_V_s0 = np.zeros(25)
    ref_V_s1 = np.zeros(25)
    ref_x_s0 = np.zeros(25)
    ref_x_s1 = np.zeros(25)

    # On lit les 24 marges du benchmark.
    for i in range(24):
        # La marge de l'heure i se trouve dans la colonne 1 de la feuille.
        margins_24[i] = float(df.iloc[i + 1, 1])

    # On lit ensuite les décisions et valeurs de référence, y compris la ligne terminale.
    for i in range(25):
        # Si la cellule est vide, on remplace par 0 pour éviter les NaN sur les décisions.
        ref_x_s0[i] = float(df.iloc[i + 1, 6]) if pd.notna(df.iloc[i + 1, 6]) else 0
        # Même logique pour les décisions quand l'état courant vaut s=1.
        ref_x_s1[i] = float(df.iloc[i + 1, 7]) if pd.notna(df.iloc[i + 1, 7]) else 0
        # Valeur de référence quand l'état courant vaut s=0.
        ref_V_s0[i] = float(df.iloc[i + 1, 8]) if pd.notna(df.iloc[i + 1, 8]) else 0
        # Pour `V(s=1)`, on garde `np.nan` si la cellule est vide, car l'absence d'information
        # est ici significative pour la comparaison ultérieure.
        ref_V_s1[i] = float(df.iloc[i + 1, 9]) if pd.notna(df.iloc[i + 1, 9]) else np.nan

    # On renvoie à la fois les données du cas test et les références attendues.
    return margins_24, demands_24, startups_24, ref_V_s0, ref_V_s1, ref_x_s0, ref_x_s1


# ---------------------------------------------------------------------------
# Fonction 3 — Solveur de base en programmation dynamique
# ---------------------------------------------------------------------------
def solve_base_dp(margins, demands, startups):
    # `T` est l'horizon temporel du problème: 24 pour le benchmark, 168 pour la semaine.
    T = len(margins)

    # `V[t, s]` = meilleure valeur future à partir de l'heure t si l'état courant vaut s.
    # Ici, s=0 signifie OFF et s=1 signifie ON.
    V = np.zeros((T + 1, 2))

    # `X[t, s]` = décision optimale à prendre à l'heure t si l'état courant vaut s.
    # Convention utilisée:
    #   1  -> démarrer depuis OFF,
    #   0  -> rester dans le même état (OFF si on est OFF, ON si on est ON),
    #  -1  -> s'arrêter depuis ON.
    X = np.zeros((T, 2), dtype=int)

    # Condition terminale si l'on arrive à la fin déjà éteint.
    V[T, 0] = 0.0
    # Condition terminale si l'on arrive à la fin allumé.
    # Ici le notebook met 0.0 dans le solveur de base simple;
    # la contrainte terminale exacte est ensuite contrôlée par la validation et le reporting.
    V[T, 1] = 0.0

    # Boucle arrière de Bellman: on remonte de la dernière heure vers la première.
    for t in range(T - 1, -1, -1):
        # Marge unitaire de l'heure t.
        m = margins[t]
        # Demande de l'heure t.
        D = demands[t]
        # Coût de démarrage si on allume à l'heure t.
        f = startups[t]

        # Cas 1: l'état courant est OFF.
        # Si on reste OFF, on ne gagne rien à l'heure t et on hérite simplement de la valeur future OFF.
        val_stay_off = V[t + 1, 0]
        # Si on démarre, on sert la demande, on gagne m*D, on paie f,
        # puis on continue demain en état ON.
        val_turn_on = m * D - f + V[t + 1, 1]

        # On choisit l'option la plus rentable entre rester éteint et démarrer.
        if val_turn_on > val_stay_off:
            # La meilleure valeur depuis (t, OFF) est celle du démarrage.
            V[t, 0] = val_turn_on
            # On mémorise la décision optimale: démarrer.
            X[t, 0] = 1
        else:
            # Sinon, mieux vaut ne pas démarrer à cette heure.
            V[t, 0] = val_stay_off
            # On mémorise la décision: rester OFF.
            X[t, 0] = 0

        # Cas 2: l'état courant est ON.
        # Si on reste allumé, on sert la demande, on encaisse la marge, puis on reste ON demain.
        val_stay_on = m * D + V[t + 1, 1]
        # Si on s'éteint immédiatement, on n'encaisse rien à l'heure t et on continue demain en OFF.
        val_turn_off = V[t + 1, 0]

        # On compare le fait de produire une heure de plus avec le fait de couper maintenant.
        if val_stay_on >= val_turn_off:
            # La meilleure valeur depuis (t, ON) est de rester allumé.
            V[t, 1] = val_stay_on
            # Décision optimale: rester ON.
            X[t, 1] = 0
        else:
            # Sinon, il est préférable de s'arrêter.
            V[t, 1] = val_turn_off
            # Décision optimale: arrêt.
            X[t, 1] = -1

    # On renvoie la fonction de valeur complète et la politique optimale.
    return V, X


# ---------------------------------------------------------------------------
# Fonction 4 — Reconstruction d'un planning à partir de la politique optimale
# ---------------------------------------------------------------------------
def extract_schedule(X, s0=0):
    # `T` = nombre d'heures de décision.
    T = X.shape[0]

    # `states[t]` contiendra l'état de la centrale au début de l'heure t.
    # La taille vaut T+1 pour inclure l'état final après la dernière décision.
    states = np.zeros(T + 1, dtype=int)

    # `decisions[t]` contiendra la décision réellement appliquée à l'heure t.
    decisions = np.zeros(T, dtype=int)

    # On fixe l'état initial, par défaut OFF.
    states[0] = s0

    # On simule ensuite le système en suivant la politique `X`.
    for t in range(T):
        # État courant au début de l'heure t.
        s = states[t]
        # Décision optimale prescrite par la politique dans cet état.
        x = X[t, s]
        # On stocke cette décision pour le reporting.
        decisions[t] = x

        # Si on était OFF et qu'on démarre, l'état suivant devient ON.
        if s == 0 and x == 1:
            states[t + 1] = 1
        # Si on était ON et qu'on s'arrête, l'état suivant devient OFF.
        elif s == 1 and x == -1:
            states[t + 1] = 0
        else:
            # Dans tous les autres cas, l'état ne change pas.
            states[t + 1] = s

    # On renvoie le chemin d'états et la suite des décisions appliquées.
    return states, decisions


# ---------------------------------------------------------------------------
# Fonction 5 — Extension 1 : durée minimale de fonctionnement
# ---------------------------------------------------------------------------
def solve_ext1_min_runtime(margins, demands, startups, min_hours=10):
    # Horizon du problème.
    T = len(margins)

    # L'état n'est plus binaire:
    #   0   -> OFF,
    #   1   -> ON depuis 1 heure après démarrage,
    #   2   -> ON depuis 2 heures,
    #   ...,
    #   min_hours -> ON avec contrainte déjà satisfaite.
    n_states = min_hours + 1

    # Table de Bellman élargie à cette nouvelle dimension d'état.
    V = np.full((T + 1, n_states), NEG_INF)

    # Politique optimale associée.
    X = np.zeros((T, n_states), dtype=int)

    # À l'horizon terminal, seul l'état OFF est explicitement autorisé.
    V[T, 0] = 0.0

    # Backward induction classique.
    for t in range(T - 1, -1, -1):
        # Paramètres économiques de l'heure t.
        m = margins[t]
        D = demands[t]
        f = startups[t]

        # Cas `state = 0` : la centrale est OFF.
        # Valeur si on reste éteint.
        val_off = V[t + 1, 0]
        # Valeur si on démarre maintenant: on paie le coût de démarrage,
        # on produit à l'heure t, puis on bascule dans l'état "ON depuis 1 heure".
        val_on = m * D - f + V[t + 1, 1]

        # Depuis OFF, on choisit entre ne rien faire et démarrer.
        if val_on > val_off:
            V[t, 0] = val_on
            X[t, 0] = 1
        else:
            V[t, 0] = val_off
            X[t, 0] = 0

        # Cas `state = 1..min_hours-1` : la centrale est ON mais n'a pas encore rempli
        # sa durée minimale de fonctionnement. L'arrêt est donc interdit.
        for k in range(1, min_hours):
            # On est forcé de produire à l'heure t et de passer au compteur suivant `k+1`.
            V[t, k] = m * D + V[t + 1, k + 1]
            # La seule décision admissible est de rester ON.
            X[t, k] = 0

        # Cas `state = min_hours` : la contrainte de durée minimale est satisfaite.
        # On retrouve la liberté de rester ON ou de s'arrêter.
        val_stay = m * D + V[t + 1, min_hours]
        val_turn_off = V[t + 1, 0]

        # On choisit entre continuer à produire ou s'arrêter.
        if val_stay >= val_turn_off:
            V[t, min_hours] = val_stay
            X[t, min_hours] = 0
        else:
            V[t, min_hours] = val_turn_off
            X[t, min_hours] = -1

    # Sortie standard: table de valeur et table de politique.
    return V, X


# ---------------------------------------------------------------------------
# Fonction 6 — Reconstruction du planning pour l'extension 1
# ---------------------------------------------------------------------------
def extract_schedule_ext1(X, min_hours=10):
    # Nombre d'heures de décision.
    T = X.shape[0]

    # `states_detail[t]` garde l'état détaillé: OFF, ON depuis 1h, ON depuis 2h, etc.
    states_detail = np.zeros(T + 1, dtype=int)
    # `states_binary[t]` simplifie ce même état en 0/1 pour les tableaux de planning.
    states_binary = np.zeros(T + 1, dtype=int)
    # `decisions[t]` mémorise la décision effectivement appliquée.
    decisions = np.zeros(T, dtype=int)

    # On déroule la politique heure par heure.
    for t in range(T):
        # État détaillé courant.
        s = states_detail[t]
        # Décision optimale prescrite pour cet état détaillé.
        x = X[t, s]
        # On la stocke pour le reporting.
        decisions[t] = x

        # Si la centrale est OFF et qu'on démarre, on passe à l'état "ON depuis 1 heure".
        if s == 0 and x == 1:
            states_detail[t + 1] = 1
        # Si la centrale est OFF et qu'on ne démarre pas, elle reste OFF.
        elif s == 0 and x == 0:
            states_detail[t + 1] = 0
        # Si elle est ON mais encore sous contrainte minimale, on incrémente simplement son âge.
        elif 1 <= s < min_hours:
            states_detail[t + 1] = s + 1
        # Si elle a atteint `min_hours` et que la politique dit d'arrêter, on repasse OFF.
        elif s == min_hours and x == -1:
            states_detail[t + 1] = 0
        else:
            # Sinon, elle reste dans l'état saturé `min_hours`,
            # qui veut dire "contrainte satisfaite et centrale toujours allumée".
            states_detail[t + 1] = min_hours

        # Conversion immédiate vers une lecture binaire simple pour le planning.
        states_binary[t + 1] = 1 if states_detail[t + 1] > 0 else 0

    # On renvoie les deux versions des états + les décisions.
    return states_binary, decisions, states_detail


# ---------------------------------------------------------------------------
# Fonction 7 — Extension 2 : plafond de production cumulé
# ---------------------------------------------------------------------------
def solve_ext2_production_cap(margins, demands, startups, max_mwh=16500, step=50):
    # Horizon temporel.
    T = len(margins)

    # Nombre de niveaux possibles pour le cumul de production.
    # Si le cap vaut 16500 et le pas 50, alors on a 331 états de cumul (de 0 à 16500 inclus).
    n_cum = max_mwh // step + 1

    # `V[t, s, c]` = meilleure valeur à partir de l'heure t,
    # avec état physique `s` et cumul déjà consommé `c` (en nombre de pas).
    V = np.full((T + 1, 2, n_cum), NEG_INF)

    # Politique optimale associée à ce nouvel espace d'état 3D.
    X = np.zeros((T, 2, n_cum), dtype=int)

    # À l'horizon final, être OFF reste autorisé quel que soit le cumul déjà consommé.
    V[T, 0, :] = 0.0
    # Être ON en fin d'horizon est interdit / très pénalisé.
    V[T, 1, :] = NEG_INF

    # Bellman backward sur le temps.
    for t in range(T - 1, -1, -1):
        # Paramètres économiques de l'heure t.
        m = margins[t]
        D = demands[t]
        f = startups[t]
        # Conversion de la demande réelle en nombre de pas de cumul.
        d_step = int(D / step)

        # Pour chaque cumul déjà consommé `c`, on calcule les décisions admissibles.
        for c in range(n_cum):
            # Si on reste OFF, le cumul ne change pas.
            val_off = V[t + 1, 0, c]
            # Si on produit à l'heure t, le cumul futur devient `new_c`.
            new_c = c + d_step
            # Si `new_c` dépasse l'espace admissible, l'action est rendue impossible via `NEG_INF`.
            val_on = m * D - f + V[t + 1, 1, new_c] if new_c < n_cum else NEG_INF

            # Depuis OFF, on choisit entre rester OFF et démarrer.
            if val_on > val_off:
                V[t, 0, c] = val_on
                X[t, 0, c] = 1
            else:
                V[t, 0, c] = val_off
                X[t, 0, c] = 0

            # Depuis ON, rester allumé consomme aussi `d_step` de cap supplémentaire.
            val_stay = m * D + V[t + 1, 1, new_c] if new_c < n_cum else NEG_INF
            # S'arrêter conserve le cumul `c` inchangé pour la suite.
            val_turn_off = V[t + 1, 0, c]

            # On choisit entre continuer à produire sous contrainte de cap ou couper.
            if val_stay >= val_turn_off:
                V[t, 1, c] = val_stay
                X[t, 1, c] = 0
            else:
                V[t, 1, c] = val_turn_off
                X[t, 1, c] = -1

    # Sortie standard du solveur.
    return V, X


# ---------------------------------------------------------------------------
# Fonction 8 — Reconstruction du planning pour l'extension 2
# ---------------------------------------------------------------------------
def extract_schedule_ext2(X, demands, step=50):
    # Nombre d'heures de décision.
    T = X.shape[0]

    # État ON/OFF reconstruit ex post.
    states = np.zeros(T + 1, dtype=int)
    # Cumul discret de production consommée après chaque heure.
    cum = np.zeros(T + 1, dtype=int)
    # Décisions effectivement appliquées.
    decisions = np.zeros(T, dtype=int)

    # Déroulage de la politique heure par heure.
    for t in range(T):
        # État physique courant.
        s = states[t]
        # Cumul déjà consommé avant la décision de l'heure t.
        c = cum[t]
        # Décision optimale dans cet état complet (temps, physique, cumul).
        x = X[t, s, c]
        decisions[t] = x
        # Consommation de cap si l'heure t est produite.
        d_step = int(demands[t] / step)

        # Démarrage depuis OFF: on passe ON et on consomme du cumul.
        if s == 0 and x == 1:
            states[t + 1] = 1
            cum[t + 1] = c + d_step
        # Marche continue: on reste ON et on consomme encore du cumul.
        elif s == 1 and x == 0:
            states[t + 1] = 1
            cum[t + 1] = c + d_step
        # Arrêt: on repasse OFF sans consommer de cumul supplémentaire.
        elif s == 1 and x == -1:
            states[t + 1] = 0
            cum[t + 1] = c
        else:
            # OFF -> OFF: ni l'état ni le cumul ne changent.
            states[t + 1] = 0
            cum[t + 1] = c

    # On renvoie le planning, les décisions et le cumul reconstruit.
    return states, decisions, cum


# ---------------------------------------------------------------------------
# Fonction 9 — Extension 3 : demande stochastique le dimanche
# ---------------------------------------------------------------------------
def solve_ext3_stochastic(margins, demands, startups, stoch_demand, max_mwh=16500, step=50):
    # Horizon temporel total.
    T = len(margins)

    # Nombre d'états de cumul admissibles sous le cap hebdomadaire.
    n_cum = max_mwh // step + 1

    # Fonction de valeur sur l'état étendu (temps, OFF/ON, cumul).
    V = np.full((T + 1, 2, n_cum), NEG_INF)

    # Politique optimale associée.
    X = np.zeros((T, 2, n_cum), dtype=int)

    # Condition terminale: OFF autorisé, ON interdit.
    V[T, 0, :] = 0.0
    V[T, 1, :] = NEG_INF

    # Bellman backward sur toutes les heures de la semaine.
    for t in range(T - 1, -1, -1):
        # Marge unitaire à l'heure t.
        m = margins[t]
        # Coût de démarrage à l'heure t.
        f = startups[t]
        # Jour courant sous forme d'indice 0..6.
        day = t // 24
        # Heure dans la journée courante, 0..23.
        hour_in_day = t % 24

        # Si on est dimanche (`day == 6`), on remplace la demande déterministe
        # par deux scénarios probabilisés issus de la feuille stochastique.
        if day == 6:
            D1, p1, D2, p2 = stoch_demand[hour_in_day]
            scenarios = [(D1, p1), (D2, p2)]
        else:
            # Tous les autres jours restent purement déterministes.
            scenarios = [(demands[t], 1.0)]

        # Calcul des valeurs pour chaque niveau de cumul déjà consommé.
        for c in range(n_cum):
            # Si on reste OFF, aucun aléa n'a d'effet: on reste simplement OFF demain.
            ev_off = V[t + 1, 0, c]

            # `ev_on` va accumuler l'espérance de valeur si on démarre à l'heure t.
            ev_on = 0.0
            # Drapeau booléen: il deviendra False si au moins un scénario dépasse le cap.
            feasible_on = True

            # On somme les contributions de tous les scénarios de demande possibles.
            for D_val, prob in scenarios:
                # Nouveau cumul si la demande réalisée vaut `D_val`.
                new_c = c + int(D_val / step)
                # Si le nouveau cumul reste admissible, on ajoute la contribution pondérée.
                if new_c < n_cum:
                    ev_on += prob * (m * D_val - f + V[t + 1, 1, new_c])
                else:
                    # Si un scénario viole le cap, l'action devient non faisable telle quelle.
                    feasible_on = False
                    break

            # Si au moins un scénario rend l'action impossible, on punit l'option.
            if not feasible_on:
                ev_on = NEG_INF

            # Depuis OFF, on choisit entre rester OFF et démarrer en espérance.
            if ev_on > ev_off:
                V[t, 0, c] = ev_on
                X[t, 0, c] = 1
            else:
                V[t, 0, c] = ev_off
                X[t, 0, c] = 0

            # `ev_stay` va accumuler l'espérance de valeur si on reste ON.
            ev_stay = 0.0
            # Même logique de faisabilité que pour le démarrage.
            feasible_stay = True

            # On calcule l'espérance conditionnelle de rester ON sous tous les scénarios.
            for D_val, prob in scenarios:
                new_c = c + int(D_val / step)
                if new_c < n_cum:
                    ev_stay += prob * (m * D_val + V[t + 1, 1, new_c])
                else:
                    feasible_stay = False
                    break

            # Si rester ON viole le cap dans au moins un scénario, on interdit l'action.
            if not feasible_stay:
                ev_stay = NEG_INF

            # Valeur de l'arrêt immédiat: on passe simplement à OFF demain sans produire maintenant.
            ev_turn_off = V[t + 1, 0, c]

            # Depuis ON, on choisit entre rester ON en espérance et s'éteindre.
            if ev_stay >= ev_turn_off:
                V[t, 1, c] = ev_stay
                X[t, 1, c] = 0
            else:
                V[t, 1, c] = ev_turn_off
                X[t, 1, c] = -1

    # On renvoie la valeur optimale et la politique correspondante.
    return V, X


# ---------------------------------------------------------------------------
# Fonction 10 — Reformatage 168 -> tableau 24 x 7
# ---------------------------------------------------------------------------
def reshape_week(values, index_name="Hour"):
    # `np.asarray` garantit qu'on travaille bien avec un tableau NumPy,
    # puis `reshape(7, 24)` coupe la série en 7 jours de 24 heures.
    arr = np.asarray(values).reshape(7, 24).T
    # Le `.T` transpose pour obtenir une lecture naturelle: lignes = heures, colonnes = jours.
    return pd.DataFrame(arr, index=pd.Index(range(1, 25), name=index_name), columns=DAYS)


# ---------------------------------------------------------------------------
# Fonction 11 — Construction de la ligne terminale de Bellman
# ---------------------------------------------------------------------------
def build_terminal_row(V):
    # On extrait la dernière ligne de la fonction de valeur pour les deux états,
    # afin de rendre la condition terminale visible dans le rendu projet.
    return pd.DataFrame({"state": [0, 1], "V_terminal": [V[-1, 0], V[-1, 1]]})


# ---------------------------------------------------------------------------
# Fonction 12 — Conversion des états en statut de planning lisible
# ---------------------------------------------------------------------------
def schedule_status(states):
    # `status[t]` produira une étiquette courte pour chaque heure:
    # `ON*` pour un démarrage, `ON` pour une marche continue, `.` pour OFF.
    status = []

    # On parcourt les transitions entre l'état au début de l'heure t et celui au début de t+1.
    for t in range(len(states) - 1):
        # Transition OFF -> ON: c'est un démarrage.
        if states[t] == 0 and states[t + 1] == 1:
            status.append("ON*")
        # Si l'état suivant vaut ON sans démarrage, la centrale est simplement en marche.
        elif states[t + 1] == 1:
            status.append("ON")
        else:
            # Sinon la centrale est arrêtée sur cette heure.
            status.append(".")

    # Retour sous forme de tableau NumPy pour faciliter les traitements en aval.
    return np.array(status)


# ---------------------------------------------------------------------------
# Fonction 13 — Calcul du profit horaire réalisé à partir d'un planning
# ---------------------------------------------------------------------------
def hourly_profit(states, margins, demands, startups):
    # Un profit horaire est défini pour chaque heure de décision, donc longueur `len(states)-1`.
    profit = np.zeros(len(states) - 1)

    # On déroule le planning reconstruit heure par heure.
    for t in range(len(states) - 1):
        # Si on démarre à l'heure t, on encaisse la marge sur la demande mais on paie le coût de démarrage.
        if states[t] == 0 and states[t + 1] == 1:
            profit[t] = margins[t] * demands[t] - startups[t]
        # Si on était déjà ON et qu'on reste ON, on encaisse seulement la marge d'exploitation.
        elif states[t + 1] == 1:
            profit[t] = margins[t] * demands[t]

    # On renvoie la série complète des profits horaires.
    return profit


# ---------------------------------------------------------------------------
# Fonction 14 — Construction du tableau horaire détaillé
# ---------------------------------------------------------------------------
def build_hourly_schedule_table(states, margins, demands, startups):
    # On convertit d'abord la trajectoire d'états en statuts lisibles (`ON*`, `ON`, `.`).
    status = schedule_status(states)
    # On calcule ensuite le profit économique réellement gagné à chaque heure.
    profit = hourly_profit(states, margins, demands, startups)

    # Liste de dictionnaires qui sera convertie en DataFrame à la fin.
    rows = []

    # On construit une ligne de reporting par heure.
    for t in range(len(status)):
        rows.append(
            {
                # Numéro d'heure global dans la semaine, de 1 à 168.
                "t": t + 1,
                # Nom du jour correspondant à cette heure.
                "day": DAYS[t // 24],
                # Heure dans la journée, de 1 à 24.
                "hour": t % 24 + 1,
                # Statut compact de fonctionnement.
                "status": status[t],
                # Marge unitaire utilisée dans le calcul économique.
                "margin_eur_per_mwh": margins[t],
                # Demande servie à cette heure.
                "demand_mw": demands[t],
                # Coût de démarrage potentiellement payé si l'heure correspond à un allumage.
                "startup_cost_eur": startups[t],
                # Profit effectivement réalisé sur cette heure selon le planning optimal.
                "hourly_profit_eur": profit[t],
            }
        )

    # Conversion finale en DataFrame Pandas pour affichage et export.
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Fonction 15 — Résumé économique d'un scénario complet
# ---------------------------------------------------------------------------
def summarize_case(states, margins, demands, startups, label):
    # On recalcule la série des profits horaires à partir du planning optimal.
    profit = hourly_profit(states, margins, demands, startups)
    # On génère aussi les statuts de planning pour compter les démarrages.
    status = schedule_status(states)
    # La production totale est la somme des demandes sur toutes les heures où la centrale est ON.
    total_mwh = float(sum(demands[t] for t in range(len(status)) if states[t + 1] == 1))
    # Le dictionnaire retourné résume le scénario dans un format directement exploitable par Pandas.
    return {
        # Nom lisible du scénario.
        "case": label,
        # Profit total sur l'horizon complet.
        "profit_eur": float(profit.sum()),
        # Production totale sur la semaine.
        "production_mwh": total_mwh,
        # Nombre total de démarrages détectés via le statut `ON*`.
        "startups": int((status == "ON*").sum()),
    }


## Cellule 4 — Chargement des données

Ici on appelle les fonctions de lecture précédemment définies.

Objectif:
- récupérer les séries hebdomadaires ;
- récupérer les données du benchmark 24h ;
- produire un premier tableau descriptif pour vérifier visuellement les ordres de grandeur.


## Chargement des données


In [3]:
# On charge les séries principales du problème hebdomadaire et les scénarios stochastiques du dimanche.
margins, demands, startups, stoch_demand = load_data()
# On charge en parallèle le cas de validation 24 heures et les résultats de référence.
margins_24, demands_24, startups_24, ref_V_s0, ref_V_s1, ref_x_s0, ref_x_s1 = load_24h_validation()

# On construit ici un DataFrame Pandas pour rendre les résultats lisibles et exportables.
overview = pd.DataFrame(
    # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
    {
        # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
        "series": ["Margins", "Demand", "Startup cost"],
        # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
        "min": [margins.min(), demands.min(), startups.min()],
        # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
        "max": [margins.max(), demands.max(), startups.max()],
        # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
        "mean": [margins.mean(), demands.mean(), startups.mean()],
    # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
    }
# Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
)
# Affiche une synthèse simple pour vérifier que les données chargées ont des ordres de grandeur plausibles.
display(overview)


,series,min,max,mean
0,Margins,-5.25,6.73,0.236964
1,Demand,100.00,400.00,211.904762
2,Startup cost,800.00,3600.00,2151.785714


## Cellule 6 — Validation sur 24 heures

Avant d'attaquer le problème complet, on vérifie que le solveur reproduit bien l'exemple de référence.

Pourquoi cette étape est indispensable:
- si le solveur échoue ici, il n'y a aucune raison de lui faire confiance sur 168 heures ;
- c'est la preuve que la récurrence de Bellman et les conventions de décision sont correctes.


## Validation sur le cas 24 heures

L'énoncé de base exige la reproduction de la valeur optimale `V1(0) ≈ 1652.83`.


En pratique, si cette étape échoue, il faut corriger le solveur avant de continuer.


In [4]:
# On exécute le solveur de base sur le benchmark 24 heures.
V_24, X_24 = solve_base_dp(margins_24, demands_24, startups_24)

# On construit ici un DataFrame Pandas pour rendre les résultats lisibles et exportables.
validation_summary = pd.DataFrame(
    # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
    {
        # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
        "metric": ["Calculated V1(0)", "Reference V1(0)", "Absolute gap"],
        # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
        "value": [V_24[0, 0], ref_V_s0[0], abs(V_24[0, 0] - ref_V_s0[0])],
    # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
    }
# Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
)

# On construit ici un DataFrame Pandas pour rendre les résultats lisibles et exportables.
compare_24h = pd.DataFrame(
    # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
    {
        # On crée ici une séquence d'indices explicites pour construire un tableau de restitution.
        "hour": np.arange(1, 25),
        # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
        "margin": margins_24,
        # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
        "x_calc_s0": X_24[:, 0],
        # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
        "x_ref_s0": ref_x_s0[:24].astype(int),
        # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
        "match_s0": X_24[:, 0] == ref_x_s0[:24].astype(int),
        # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
        "V_calc_s0": V_24[:24, 0],
        # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
        "V_ref_s0": ref_V_s0[:24],
    # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
    }
# Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
)

# Affiche le résumé numérique de la validation du solveur.
display(validation_summary)
# Affiche la comparaison détaillée heure par heure entre calcul et référence.
display(compare_24h)


,metric,value
0,Calculated V1(0),1652.833977
1,Reference V1(0),1652.833977
2,Absolute gap,0.000000


,hour,margin,x_calc_s0,x_ref_s0,match_s0,V_calc_s0,V_ref_s0
0,1,-1.837175,0,0,True,1652.833977,1652.833977
1,2,4.103765,1,1,True,1652.833977,1652.833977
2,3,1.580391,0,0,True,1284.820767,1284.820767
3,4,0.902039,0,0,True,1284.820767,1284.820767
4,5,-2.906063,0,0,True,1284.820767,1284.820767
5,6,2.766988,1,1,True,1284.820767,1284.820767
6,7,-1.100000,0,0,True,1153.121988,1153.121988
7,8,-0.350000,0,0,True,1153.121988,1153.121988
8,9,2.489049,1,1,True,1153.121988,1153.121988
9,10,2.414172,1,1,True,904.217039,904.217039


## Cellule 8 — Résolution du cas de base sur 168 heures

Cette cellule applique le solveur de base au jeu de données hebdomadaire complet.

Elle produit:
- la matrice de valeur `V_base` ;
- la politique optimale `X_base` ;
- la trajectoire d'états reconstruite ;
- un résumé économique du scénario de base.


## Problème de base sur 168 heures


In [5]:
# On résout maintenant le problème de base complet sur 168 heures.
V_base, X_base = solve_base_dp(margins, demands, startups)
# On reconstruit le planning optimal à partir des décisions stockées dans `X_base`.
states_base, decisions_base = extract_schedule(X_base, s0=0)

# On construit ici un DataFrame Pandas pour rendre les résultats lisibles et exportables.
base_summary = pd.DataFrame([summarize_case(states_base, margins, demands, startups, "Base 168h")])
# Affiche les indicateurs principaux du cas de base.
display(base_summary)


,case,profit_eur,production_mwh,startups
0,Base 168h,20699.0,19000.0,8


## Cellule 10 — Tables du cas de base

Une fois le solveur exécuté, on transforme les sorties brutes en objets lisibles.

Objectif:
- afficher les décisions optimales par état ;
- afficher les fonctions de valeur ;
- rendre visible la condition terminale du problème.

C'est la passerelle entre le calcul dynamique et la restitution projet.


### Tableaux demandés pour le cas de base

Les matrices ci-dessous donnent les décisions optimales et les fonctions de valeur sur les 168 heures, organisées en grille `24 x 7`.


Ces tables sont exactement le type d'objet qu'on veut voir dans un rendu de projet: elles rendent la politique optimale auditable.


In [6]:
# Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
base_x_s0 = reshape_week(X_base[:, 0])
# Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
base_x_s1 = reshape_week(X_base[:, 1])
# Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
base_V_s0 = reshape_week(np.round(V_base[:168, 0], 2))
# Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
base_V_s1 = reshape_week(np.round(V_base[:168, 1], 2))
# Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
base_terminal = build_terminal_row(V_base)

# Ajoute un titre ou un commentaire formaté dans la sortie du notebook pour guider la lecture.
display(Markdown("**Decisions optimales x_t(s=0)**"))
# Affiche la table des décisions optimales quand l'état courant est `OFF`.
display(base_x_s0)
# Ajoute un titre ou un commentaire formaté dans la sortie du notebook pour guider la lecture.
display(Markdown("**Decisions optimales x_t(s=1)**"))
# Affiche la table des décisions optimales quand l'état courant est `ON`.
display(base_x_s1)
# Ajoute un titre ou un commentaire formaté dans la sortie du notebook pour guider la lecture.
display(Markdown("**Fonctions de valeur V_t(s=0)**"))
# Affiche la fonction de valeur associée à l'état `OFF`.
display(base_V_s0)
# Ajoute un titre ou un commentaire formaté dans la sortie du notebook pour guider la lecture.
display(Markdown("**Fonctions de valeur V_t(s=1)**"))
# Affiche la fonction de valeur associée à l'état `ON`.
display(base_V_s1)
# Ajoute un titre ou un commentaire formaté dans la sortie du notebook pour guider la lecture.
display(Markdown("**Condition terminale**"))
# Affiche explicitement la condition terminale du problème dynamique.
display(base_terminal)


**Decisions optimales x_t(s=0)**

,Lun,Mar,Mer,Jeu,Ven,Sam,Dim
Hour,,,,,,,
1,0,0,1,0,0,0,0
2,0,0,0,0,0,1,0
3,0,0,1,0,0,0,0
4,0,0,0,0,0,0,0
5,0,0,0,0,0,1,0
6,0,0,0,1,0,0,0
7,0,0,1,1,0,1,0
8,0,0,0,0,0,1,1
9,0,0,0,0,0,0,1


**Decisions optimales x_t(s=1)**

,Lun,Mar,Mer,Jeu,Ven,Sam,Dim
Hour,,,,,,,
1,0,0,0,0,0,0,-1
2,0,0,0,-1,0,0,-1
3,0,0,0,0,0,0,-1
4,-1,-1,0,0,0,0,-1
5,0,0,0,0,0,0,-1
6,-1,0,0,0,-1,0,0
7,0,0,0,0,-1,0,0
8,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0


**Fonctions de valeur V_t(s=0)**

,Lun,Mar,Mer,Jeu,Ven,Sam,Dim
Hour,,,,,,,
1,20699.0,15995.5,15030.5,11665.5,6195.0,6134.5,2708.5
2,20699.0,15995.5,14839.5,11665.5,6195.0,6134.5,2708.5
3,20699.0,15995.5,14839.5,11665.5,6195.0,6076.5,2708.5
4,20699.0,15995.5,14811.5,11665.5,6195.0,6076.5,2708.5
5,20699.0,15995.5,14811.5,11665.5,6195.0,6076.5,2708.5
6,20699.0,15995.5,14811.5,11665.5,6195.0,5780.5,2708.5
7,20699.0,15995.5,14811.5,10823.5,6195.0,5780.5,2708.5
8,20699.0,15995.5,12404.5,10812.5,6195.0,5230.0,2708.5
9,20699.0,15995.5,12404.5,10812.5,6195.0,4582.0,2560.0


**Fonctions de valeur V_t(s=1)**

,Lun,Mar,Mer,Jeu,Ven,Sam,Dim
Hour,,,,,,,
1,21100.0,16858.0,16030.5,12757.5,7072.5,6934.0,2708.5
2,21108.0,16661.5,15758.5,11665.5,7457.5,7334.5,2708.5
3,20863.0,16474.0,15839.5,11665.5,8182.5,7134.5,2708.5
4,20699.0,15995.5,15495.5,12501.5,8227.5,6878.5,2708.5
5,20705.0,16142.5,15531.5,12543.5,7267.5,7276.5,2708.5
6,20699.0,17357.5,15651.5,13165.5,6195.0,7490.5,3748.0
7,21207.0,17727.5,16311.5,12323.5,6195.0,7580.5,4451.5
8,21069.0,16725.5,14351.5,12163.5,7007.0,7030.0,5108.5
9,21421.5,17955.5,13379.5,12790.5,6278.0,5551.0,4960.0


**Condition terminale**

,state,V_terminal
0,0,0.000000e+00
1,1,-1.000000e+15


In [7]:
# Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
base_planning = reshape_week(schedule_status(states_base))
# Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
base_hourly_table = build_hourly_schedule_table(states_base, margins, demands, startups)

# Ajoute un titre ou un commentaire formaté dans la sortie du notebook pour guider la lecture.
display(Markdown("**Planning compact ON/OFF**"))
# Affiche l'objet calculé afin de l'inspecter directement dans le notebook.
display(base_planning)
# Ajoute un titre ou un commentaire formaté dans la sortie du notebook pour guider la lecture.
display(Markdown("**Table horaire détaillée**"))
# Affiche l'objet calculé afin de l'inspecter directement dans le notebook.
display(base_hourly_table)


**Planning compact ON/OFF**

,Lun,Mar,Mer,Jeu,Ven,Sam,Dim
Hour,,,,,,,
1,.,ON,ON,ON,ON,ON,.
2,.,ON,ON,.,ON,ON,.
3,.,ON,ON,.,ON,ON,.
4,.,.,ON,.,ON,ON,.
5,.,.,ON,.,ON,ON,.
6,.,.,ON,ON*,.,ON,.
7,.,.,ON,ON,.,ON,.
8,.,.,ON,ON,.,ON,ON*
9,.,.,ON,ON,.,ON,ON


**Table horaire détaillée**

,t,day,hour,status,margin_eur_per_mwh,demand_mw,startup_cost_eur,hourly_profit_eur
0,1,Lun,1,.,-0.08,100.0,1000.0,0.0
1,2,Lun,2,.,2.45,100.0,1000.0,0.0
2,3,Lun,3,.,1.64,100.0,1000.0,0.0
3,4,Lun,4,.,-5.14,100.0,1000.0,0.0
4,5,Lun,5,.,0.06,100.0,1000.0,0.0
...,...,...,...,...,...,...,...,...
163,164,Dim,20,ON,3.96,200.0,2000.0,792.0
164,165,Dim,21,ON,0.16,200.0,2000.0,32.0
165,166,Dim,22,.,-1.48,150.0,1500.0,0.0
166,167,Dim,23,.,-2.19,150.0,1500.0,0.0


## Cellule 13 — Extensions

On passe du modèle minimal aux variantes plus réalistes.

Les trois extensions étudiées sont:
- une durée minimale de fonctionnement ;
- un plafond de production cumulé ;
- une demande stochastique le dimanche.

L'idée-clé est toujours la même: enrichir l'état du Bellman pour intégrer la nouvelle contrainte.


## Extensions


Chaque extension répond à une intuition industrielle différente: engagement minimal, rareté de production, puis incertitude.


In [8]:
# On résout l'extension 1 avec une durée minimale de fonctionnement de 10 heures.
V_ext1, X_ext1 = solve_ext1_min_runtime(margins, demands, startups, min_hours=10)
# On reconstruit la trajectoire optimale de l'extension 1, à la fois en version binaire et détaillée.
states_ext1, decisions_ext1, states_detail_ext1 = extract_schedule_ext1(X_ext1, min_hours=10)

# On résout l'extension 2 avec un plafond hebdomadaire de production.
V_ext2, X_ext2 = solve_ext2_production_cap(margins, demands, startups, max_mwh=16500)
# On reconstitue le planning et le cumul de production de l'extension 2.
states_ext2, decisions_ext2, cum_ext2 = extract_schedule_ext2(X_ext2, demands, step=50)

# On résout l'extension 3, où le dimanche devient stochastique sous contrainte de cap.
V_ext3, X_ext3 = solve_ext3_stochastic(margins, demands, startups, stoch_demand, max_mwh=16500)
# On crée une copie des demandes déterministes pour construire le scénario proxy basé sur l'espérance de demande.
demands_expected = demands.copy()
# Cette boucle parcourt systématiquement une dimension importante du problème pour remplir ou lire les structures de calcul.
for h in range(24):
    # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
    D1, p1, D2, p2 = stoch_demand[h]
    # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
    demands_expected[144 + h] = D1 * p1 + D2 * p2
# On résout le proxy déterministe en remplaçant l'aléa du dimanche par la demande moyenne `E[D]`.
V_ext3_det, _ = solve_ext2_production_cap(margins, demands_expected, startups, max_mwh=16500)

# Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
summary_rows = [
    # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
    summarize_case(states_base, margins, demands, startups, "Base"),
    # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
    summarize_case(states_ext1, margins, demands, startups, "Ext 1 - min 10h"),
    # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
    summarize_case(states_ext2, margins, demands, startups, "Ext 2 - cap 16500"),
    # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
    {
        # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
        "case": "Ext 3 - stochastic + cap",
        # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
        "profit_eur": float(V_ext3[0, 0, 0]),
        # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
        "production_mwh": np.nan,
        # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
        "startups": np.nan,
    # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
    },
    # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
    {
        # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
        "case": "Ext 3 with E[D]",
        # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
        "profit_eur": float(V_ext3_det[0, 0, 0]),
        # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
        "production_mwh": np.nan,
        # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
        "startups": np.nan,
    # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
    },
# Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
]
# On construit ici un DataFrame Pandas pour rendre les résultats lisibles et exportables.
extensions_summary = pd.DataFrame(summary_rows)
# On récupère ici une valeur de référence dans un tableau déjà construit pour calculer un écart.
extensions_summary["delta_vs_base"] = extensions_summary["profit_eur"] - extensions_summary.loc[0, "profit_eur"]
# Affiche le tableau comparatif final entre le cas de base et les extensions.
display(extensions_summary)


,case,profit_eur,production_mwh,startups,delta_vs_base
0,Base,20699.000000,19000.0,8.0,0.000000
1,Ext 1 - min 10h,19785.000000,18800.0,6.0,-914.000000
2,Ext 2 - cap 16500,19764.000000,16400.0,8.0,-935.000000
3,Ext 3 - stochastic + cap,19459.218628,NaN,NaN,-1239.781372
4,Ext 3 with E[D],19764.000000,NaN,NaN,-935.000000


In [9]:
# Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
ext1_planning = reshape_week(schedule_status(states_ext1))
# Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
ext2_planning = reshape_week(schedule_status(states_ext2))
# On crée ici une séquence d'indices explicites pour construire un tableau de restitution.
ext2_cum_table = pd.DataFrame({"t": np.arange(1, 169), "cum_step_50": cum_ext2[1:], "cum_mwh": cum_ext2[1:] * 50})

# Ajoute un titre ou un commentaire formaté dans la sortie du notebook pour guider la lecture.
display(Markdown("**Planning extension 1**"))
# Affiche le planning hebdomadaire de l'extension 1 sous forme compacte.
display(ext1_planning)
# Ajoute un titre ou un commentaire formaté dans la sortie du notebook pour guider la lecture.
display(Markdown("**Planning extension 2**"))
# Affiche le planning hebdomadaire de l'extension 2 sous forme compacte.
display(ext2_planning)
# Ajoute un titre ou un commentaire formaté dans la sortie du notebook pour guider la lecture.
display(Markdown("**Cumul de production extension 2**"))
# Affiche le cumul de production de l'extension 2 heure par heure.
display(ext2_cum_table)

# Ajoute un titre ou un commentaire formaté dans la sortie du notebook pour guider la lecture.
display(Markdown("**Note extension 3**"))
# Ajoute un titre ou un commentaire formaté dans la sortie du notebook pour guider la lecture.
display(Markdown(
    # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
    "Avec demande stochastique le dimanche, le résultat est une politique conditionnelle et non un planning fixe unique. "
    # Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
    f"Le profit espéré optimal vaut **{V_ext3[0, 0, 0]:.2f} EUR**, contre **{V_ext3_det[0, 0, 0]:.2f} EUR** si l'on remplace la demande aléatoire par son espérance."
# Cette ligne participe directement à la construction du solveur, du reporting, ou de l'export du notebook.
))


**Planning extension 1**

,Lun,Mar,Mer,Jeu,Ven,Sam,Dim
Hour,,,,,,,
1,.,ON,ON,ON,ON,ON,.
2,.,ON,ON,ON,ON,ON,.
3,.,ON,ON,ON,ON,ON,.
4,.,.,ON,ON,ON,ON,.
5,.,.,ON,ON,ON,ON,.
6,.,.,ON,ON,.,ON,.
7,.,.,ON,ON,.,ON,.
8,.,.,ON,ON,.,ON,ON*
9,.,.,ON,ON,.,ON,ON


**Planning extension 2**

,Lun,Mar,Mer,Jeu,Ven,Sam,Dim
Hour,,,,,,,
1,.,ON,ON,ON,.,.,.
2,.,ON,ON,.,.,.,.
3,.,ON,ON,.,.,.,.
4,.,.,ON,.,.,.,.
5,.,.,ON,.,.,ON*,.
6,.,.,ON,ON*,.,ON,.
7,.,.,ON,ON,.,ON,.
8,.,.,ON,ON,.,ON,ON*
9,.,.,ON,ON,.,ON,ON


**Cumul de production extension 2**

,t,cum_step_50,cum_mwh
0,1,0,0
1,2,0,0
2,3,0,0
3,4,0,0
4,5,0,0
...,...,...,...
163,164,328,16400
164,165,328,16400
165,166,328,16400
166,167,328,16400


**Note extension 3**

Avec demande stochastique le dimanche, le résultat est une politique conditionnelle et non un planning fixe unique. Le profit espéré optimal vaut **19459.22 EUR**, contre **19764.00 EUR** si l'on remplace la demande aléatoire par son espérance.

## Cellule 16 — Export final

Dernière étape: transformer les objets calculés dans le notebook en livrables Excel.

Pourquoi c'est utile:
- les résultats deviennent partageables ;
- on peut remettre un fichier propre sans relancer tout le notebook à la main ;
- chaque feuille correspond à un aspect analytique du projet.


## Export des résultats

Le classeur ci-dessous facilite la remise des tableaux de résultats.


Le notebook ne s'arrête donc pas au calcul: il prépare aussi la remise finale.


In [10]:
# On ouvre un writer Excel pour exporter proprement toutes les tables produites.
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    # Cette ligne écrit un tableau de résultats dans une feuille dédiée du fichier Excel de sortie.
    validation_summary.to_excel(writer, sheet_name="Validation24h", index=False)
    # Cette ligne écrit un tableau de résultats dans une feuille dédiée du fichier Excel de sortie.
    compare_24h.to_excel(writer, sheet_name="Validation24h_Compare", index=False)
    # Cette ligne écrit un tableau de résultats dans une feuille dédiée du fichier Excel de sortie.
    base_summary.to_excel(writer, sheet_name="BaseSummary", index=False)
    # Cette ligne écrit un tableau de résultats dans une feuille dédiée du fichier Excel de sortie.
    base_x_s0.to_excel(writer, sheet_name="Base_X_s0")
    # Cette ligne écrit un tableau de résultats dans une feuille dédiée du fichier Excel de sortie.
    base_x_s1.to_excel(writer, sheet_name="Base_X_s1")
    # Cette ligne écrit un tableau de résultats dans une feuille dédiée du fichier Excel de sortie.
    base_V_s0.to_excel(writer, sheet_name="Base_V_s0")
    # Cette ligne écrit un tableau de résultats dans une feuille dédiée du fichier Excel de sortie.
    base_V_s1.to_excel(writer, sheet_name="Base_V_s1")
    # Cette ligne écrit un tableau de résultats dans une feuille dédiée du fichier Excel de sortie.
    base_terminal.to_excel(writer, sheet_name="Base_Terminal", index=False)
    # Cette ligne écrit un tableau de résultats dans une feuille dédiée du fichier Excel de sortie.
    base_planning.to_excel(writer, sheet_name="Base_Planning")
    # Cette ligne écrit un tableau de résultats dans une feuille dédiée du fichier Excel de sortie.
    base_hourly_table.to_excel(writer, sheet_name="Base_Hourly", index=False)
    # Cette ligne écrit un tableau de résultats dans une feuille dédiée du fichier Excel de sortie.
    ext1_planning.to_excel(writer, sheet_name="Ext1_Planning")
    # Cette ligne écrit un tableau de résultats dans une feuille dédiée du fichier Excel de sortie.
    ext2_planning.to_excel(writer, sheet_name="Ext2_Planning")
    # Cette ligne écrit un tableau de résultats dans une feuille dédiée du fichier Excel de sortie.
    ext2_cum_table.to_excel(writer, sheet_name="Ext2_Cum", index=False)
    # Cette ligne écrit un tableau de résultats dans une feuille dédiée du fichier Excel de sortie.
    extensions_summary.to_excel(writer, sheet_name="Extensions_Summary", index=False)

# Message final confirmant l'emplacement du fichier exporté.
print(f"Results exported to: {OUTPUT_XLSX}")


Results exported to: c:\Users\3733KN\Documents\Perso\realoption\realoption-p3\dispatch_results.xlsx
